# The confidence check (slice M3): calibration, from the run log

For a patient who is not starting to deteriorate, at most 1 in 100 of their 15-minute checks (alpha = 0.01) should show a
supported alert, on average over patients. The check calibrates one score quantile *q* on one randomly drawn negative bin
per stay, then calls a student alert supported only when p is above both the student's threshold *t* and *q*. This
notebook only reads the run log (`training/runs/confidence_check/`) and, when it exists, the held-out text file that was
written once. It computes no new number. **Synthetic, not validated.**

In [ ]:
# Finds the repository root (the folder that holds models/ and training/). This notebook only reads and draws.
import json
import logging
import re
from pathlib import Path

logging.getLogger("matplotlib").setLevel(logging.ERROR)
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib.ticker import MaxNLocator

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "models").is_dir() and (p / "training").is_dir())
COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]  # Okabe-Ito
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 9})


def ints(*axes):
    """Whole numbers on the x axis (generations and epochs)."""
    for ax in axes:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))


def newest_log(model):
    """The run log with the newest file name under training/runs/<model>/ (the names carry the date)."""
    logs = sorted((ROOT / "training" / "runs" / model).glob("*.json"))
    return json.loads(logs[-1].read_text()), logs[-1].name


In [ ]:
log, log_name = newest_log("confidence_check")
CC = ROOT / "models" / "tier_c" / "confidence_check"
LABEL = {"all": "with temperature", "no_temp": "without temperature"}
print(f"run log {log_name}: run id {log['run_id']}, commit {log['commit']}")
print(f"alpha {log['alpha']}, draw seed {log['draw_seed']}, calibration seeds {log['calibration_seeds'][0]}-{log['calibration_seeds'][-1]} ({len(log['calibration_seeds'])} trials)")
for tag, why in log["not_applicable"].items():
    print(f"{LABEL.get(tag, tag)}: not applicable ({why}); no calibration")

## The calibration scores

The student's score for one drawn negative bin per stay, sorted (the run log keeps at most 200 evenly spaced order statistics). *q* is the k-th smallest of the n drawn scores, k = n + 1 - j with j = floor(alpha (n + 1)). *t* is the student's alert threshold. If q is below t the check is not binding and removes no alert.

In [ ]:
tags = list(log["variants"])
fig, axes = plt.subplots(1, max(len(tags), 1), figsize=(6.2 * max(len(tags), 1), 4), squeeze=False)
for ax, tag in zip(axes[0], tags):
    v = log["variants"][tag]
    s = v["score_quantiles"]
    ax.plot([i / (len(s) - 1) if len(s) > 1 else 0 for i in range(len(s))], [max(x, 1e-12) for x in s], "-", color="#0072B2", label="sorted calibration scores")
    ax.axhline(v["t"], color="#009E73", ls="--", label=f"t = {v['t']}")
    if v["q"] is not None:
        ax.axhline(v["q"], color="#D55E00", ls=":", label=f"q = {v['q']}")
    ax.set_yscale("log")
    ax.set_title(f"Calibration scores, {LABEL.get(tag, tag)}")
    ax.set_xlabel("position in the sorted scores (0 smallest, 1 largest)")
    ax.set_ylabel("student p (log scale)")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
for tag in tags:
    v = log["variants"][tag]
    q = "infinity (j = 0: nothing is supported)" if v["q"] is None else v["q"]
    print(f"Reading ({LABEL.get(tag, tag)}): n = {v['n']} stays drawn from {v['negative_bins']} negative bins, j = {v['j']}, k = {v['k']}, q = {q}, t = {v['t']}; "
          + ("binding: the check can remove alerts." if v["binding"] else "not binding (q is below t): the check removes no alert."))

## The held-out result

Written once by `research.confidence evaluate` on ward-v2 seeds 11-30 into `heldout.txt`. The lines below are quoted from that file: the achieved error against alpha, the share of bins that are "not sure", the alerting bins and the episodes the check loses.

In [ ]:
held = CC / "heldout.txt"
if not held.exists():
    print("heldout.txt has not been written yet")
else:
    keep = re.compile(r"^(\[\w+\] confidence check|calibration facts|binding|NOT binding|\d\. |\s+(context|verdict):)")
    for line in held.read_text().splitlines():
        if keep.match(line):
            print(line)